In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
%cd /content/drive/MyDrive/fload_lstm

/content/drive/MyDrive/fload_lstm


In [20]:
file_path = "FLOAD_데이터_현황 - 침수이력_API전체.csv"
df = pd.read_csv(file_path, header=3, encoding='utf-8')

In [21]:
# LSTM에 사용할 핵심 열만 추출 (시간, 공간, 침수정도)
cols_to_use = ['발생일자', '시군구', '침수심(cm)', '침수면적(m2)']
df_lstm = df[cols_to_use].copy()

In [22]:
df_lstm['침수면적(m2)'] = df_lstm['침수면적(m2)'].astype(str).str.replace(',', '').astype(float)
df_lstm['침수심(cm)'] = df_lstm['침수심(cm)'].fillna(0)

In [23]:
df_lstm['발생일자'] = pd.to_datetime(df_lstm['발생일자'])
df_lstm = df_lstm.sort_values('발생일자').reset_index(drop=True)

print(" 데이터 전처리 완료. 데이터 샘플:")
print(df_lstm.head())

 데이터 전처리 완료. 데이터 샘플:
        발생일자   시군구  침수심(cm)  침수면적(m2)
0 2009-07-16  해운대구     40.0  112438.0
1 2009-07-16   연제구    102.0   90078.8
2 2009-07-16   연제구     20.0  130117.9
3 2009-07-16   연제구     40.0  130117.9
4 2009-07-16   연제구     64.0  130117.9


In [24]:
# 2. 시계열 데이터셋 구성 (Windowing)
# ==========================================
# 수치형 데이터 스케일링 (0~1 사이로 정규화)
scaler = MinMaxScaler()
features = ['침수심(cm)', '침수면적(m2)']
df_lstm[features] = scaler.fit_transform(df_lstm[features])

# LSTM 입력을 위한 슬라이딩 윈도우(Sliding Window) 함수
# 예: 과거 5일의 데이터를 보고 다음 날의 침수 위험도 예측
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length][0]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

sequence_length = 5
data_values = df_lstm[features].values
X, y = create_sequences(data_values, sequence_length)

# PyTorch 텐서로 변환
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

In [25]:
class FloodDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = FloodDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

In [26]:
# 3. PyTorch LSTM 모델 아키텍처 정의
# ==========================================
class FloodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(FloodLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM 레이어 (batch_first=True로 설정하여 입력 형태를 [배치, 시퀀스, 특성]으로 맞춤)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # 최종 출력을 위한 Fully Connected 레이어
        self.fc = nn.Linear(hidden_size, output_size)
        # 이진 분류(침수 여부)라면 Sigmoid를, 수치 예측(침수심)이라면 바로 출력합니다.
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 은닉 상태(Hidden State)와 셀 상태(Cell State) 초기화
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # LSTM 통과
        out, _ = self.lstm(x, (h0, c0))

        # 마지막 타임스텝의 출력만 사용
        out = out[:, -1, :]

        # 최종 예측값 도출
        out = self.fc(out)
        out = self.sigmoid(out) # 예측값이 확률(0~1)이라면 사용
        return out

In [27]:
# 4. 모델 초기화 및 학습 준비
# ==========================================
INPUT_SIZE = 2    # 사용된 특성 수 (침수심, 침수면적)
HIDDEN_SIZE = 64  # LSTM 내부 노드 수
NUM_LAYERS = 2    # LSTM 층의 개수
OUTPUT_SIZE = 1   # 최종 예측할 값 (침수 확률)

model = FloodLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE)

# 손실 함수 및 최적화 기법 설정
criterion = nn.BCELoss() # 이진 분류용 Loss (침수 0 or 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(" 모델 아키텍처:")
print(model)

 모델 아키텍처:
FloodLSTM(
  (lstm): LSTM(2, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
